## Merging the Risk Hex Bins
#### Taking all the Hex GPKG's with geographic risk in them and merging together

In [1]:
import geopandas as gpd
import pandas as pd
from functools import reduce

pd.set_option('display.max_columns', None)

In [ ]:
#Calling specific ones in a folder of Hex Edits - 
#Could also call all files in a specific folder.
files = [
    "",
    "",
    "",
    "",
    "",
    "",
    ""
]

# Read each into a GeoDataFrame
gdfs = [gpd.read_file(f) for f in files]

# Merge all on 'hex_ID'
gdf_merged = gdfs[0]

for gdf in gdfs[1:]:
    # Drop geometry before merge to avoid duplication
    gdf_no_geom = gdf.drop(columns=gdf.geometry.name)
    
    gdf_merged = gdf_merged.merge(gdf_no_geom, on="h3_ID", how="outer")

# Ensure it's still a GeoDataFrame
gdf_merged = gpd.GeoDataFrame(gdf_merged, geometry="geometry")


num_cols = gdf_merged.select_dtypes(include="number").columns
gdf_merged[num_cols] = gdf_merged[num_cols].fillna(0)

gdf_merged = gdf_merged.loc[:, ~gdf_merged.columns.str.startswith("geometry_")]

gdf_merged = gdf_merged.loc[:, ~gdf_merged.columns.str.endswith(("_x", "_y"))]

#list all column headers, for sorting
column_names_list = gdf_merged.columns.tolist()
pd.Series(column_names_list).to_csv("columns2.csv", index=False)

#name and save where H3 edits are going
gdf_merged.to_file(".gpkg", driver="GPKG")

### Making a Risk Index

In [ ]:
#making the index for risk 
#read what was just made 
hex_path = ".gpkg"
#Define GPD:
gdf_merged = gpd.read_file(hex_path)

In [ ]:
cols = [
    "Wildfire_Index",
    "Flood_Index",
    "Dam_Index",
    "Sea_Index",
    "Heat_Index",
    "PA_Index",
    "event_count" 
]

# Ensure numeric (invalid values → NaN)
gdf_merged["Total"] = (
    gdf_merged[cols]
    .sum(axis=1, min_count=1)   # keeps NaN if all inputs are NaN
    .fillna(0)                  # replace all-NaN rows with 0
    .round()
    .astype("Int64")            # nullable integer
)

gdf_merged["Total"] = gdf_merged["Total"].round().astype("Int64")

# Min-max normalization
min_val = gdf_merged["Total"].min()
max_val = gdf_merged["Total"].max()

gdf_merged["Total_Index"] = (gdf_merged["Total"] - min_val) / (max_val - min_val)
print(gdf_merged["Total_Index"].describe())

#name and save where H3 edits are going -overwrite with additions.
gdf_merged.to_file(".gpkg", driver="GPKG")